# 4. 行情

**行情 (Quote)** 是指市場某一特定時點的資產價格相關資訊，例如買價和賣價等。

Eskmo 中行情分為 **即時行情** 與 **歷史行情**，兩者操作方式不同。

並且在不同商品別（證券、期貨等）之下，提供的行情資訊也略有不同。

------

## 4.1. 即時行情

即時行情包含 **行情 (Quote)**, **成交明細 (Tick)**, **最佳五檔 (Best5)** 與 **K 線 (KLine)**

### 4.1.1. 行情 (Quote)

在最近一檔價量、成交價與成交量三者其中一個產生變動時，就會有行情數據，行情不會包含詳細的成交明細與五檔資訊。

在訂閱商品檔行情時，就會主動觸發一筆當前最新行情，後續每次收到的即時行情，部分沒有改變的欄位會以 0 提供。

In [1]:
user_id = "A123456789"
password = "*************"

In [2]:
from eskmo import api
from eskmo import Stock, SubscribeStartResult, SubscribeFailResult, SubscribeSuccessResult

api.logger.show = False

# 註冊事件
@api.event.quote.subscribe_start
def onSubscribeStart(data: SubscribeStartResult):
    print(f"[onSubscribeStart] symbol: {data.symbol}")

@api.event.quote.subscribe_fail
def onSubscribeFail(data: SubscribeFailResult):
    print(f"[onSubscribeFail] error_code: {data}")

@api.event.quote.subscribe_success
def onSubscribeSuccess(data: SubscribeSuccessResult):
    print(f"[onSubscribeSuccess] Messages: {data.messages}")

# 4.1.2. 透過裝飾器獲得事件通知
@api.event.quote.bidask_changed
def onBidAskChanged(data):
    print(f"[ALL] Quote:: {data}")

@api.event.quote.price_changed
def onPriceChanged(data):
    print(f"onPriceChanged:: Quote:: {data}")

@api.event.quote.tick_changed
def onTickChanged(data):
    print(f"onTickChanged:: Quote:: {data}")   

isLogined = False
@api.event.user.login_success
def onLoginSuccess(data):
    global isLogined
    print("Login Success!")
    isLogined = True


api.init()
api.login(userId=user_id, password=password)

stock: Stock = api.stocks["2330"]
stock.subscribe_quote() 

api.keepalive() # 為了看到報價, 不讓 Notebook 停止

%CP INFO  2024-08-16 11:49:38.890398 | Main | 15916 |[ eskmo version: 0.0.88 ]
Login Success!
[onSubscribeStart] symbol: 2330
[onSubscribeStart] symbol: 2330
[onSubscribeSuccess] Messages: ['訂閱現股 2330 行情成功']
[onSubscribeSuccess] Messages: ['訂閱現股 2330 行情成功']
onPriceChanged:: Quote:: Quote(idx=(0, 29609), market='Stock', decimal=2, sector=24, symbol='2330', name='台積電', high=0.0, open=0.0, low=0.0, close=965.0, tick_qty=2, ref=943.0, bid=964.0, bid_qty=285, ask=965.0, ask_qty=2213, bid_total_qty=18014, ask_total_qty=9905, future_oi=0, qty_total=27919, qty_yesterday=19434, up=1035.0, down=849.0, simulate=False, day_trade_type=2, trading_day=20240816)
[ALL] Quote:: Quote(idx=(0, 29609), market='Stock', decimal=2, sector=24, symbol='2330', name='台積電', high=965.0, open=963.0, low=958.0, close=965.0, tick_qty=2, ref=943.0, bid=964.0, bid_qty=285, ask=965.0, ask_qty=2213, bid_total_qty=18014, ask_total_qty=9905, future_oi=0, qty_total=27919, qty_yesterday=19434, up=1035.0, down=849.0, simulate=

### 4.1.2. 成交明細 (Tick)

訂閱成交明細與行情相同，訂閱事件回傳即為成交明細物件 (Tick)

In [1]:
user_id = "A123456789"
password = "*************"

In [2]:
from eskmo import api
from eskmo import Stock, Tick

api.logger.show = False

@api.event.tick.notify
def onTickNotify(tick: Tick):
    print(f"[ALL] Tick:: {tick.date_str} {tick.time_str} BID: {tick.bid}, QTY: {tick.qty}")

api.init()
api.login(userId=user_id, password=password)

stock: Stock = api.stocks["2609"]
stock.subscribe_tick() 

api.keepalive()

%BP INFO  2024-08-16 11:52:17.547540 | Main | 15512 |[ eskmo version: 0.0.88 ]
[ALL] Tick:: 2024/08/16 11:52:52 BID: 64.1, QTY: 1
[ALL] Tick:: 2024/08/16 11:52:59 BID: 64.1, QTY: 3
[ALL] Tick:: 2024/08/16 11:53:04 BID: 64.1, QTY: 40
[ALL] Tick:: 2024/08/16 11:53:04 BID: 64.1, QTY: 15
[ALL] Tick:: 2024/08/16 11:53:33 BID: 64.0, QTY: 1


### 4.1.3. 最佳五檔 (Best5)

訂閱最佳五檔與成交明細、行情相同，訂閱事件回傳即為五檔物件 (Best5)

In [1]:
user_id = "A123456789"
password = "*************"

In [2]:
from eskmo import api
from eskmo import Stock, Best5

api.logger.show = False

@api.event.best5.notify
def onBest5Notify(best5: Best5):
    print(f"[ALL] Best5:: {best5.bid[0].price}, {best5.bid[0].qty}, {best5}")

api.init()
api.login(userId=user_id, password=password)

stock: Stock = api.stocks["2609"]
stock.subscribe_tick() 

api.keepalive()

%AP INFO  2024-08-16 11:53:56.053103 | Main | 39584 |[ eskmo version: 0.0.88 ]
[ALL] Best5:: 64.0, 427, Best5(code='2609', bid=[OrderBookLevel(price=64.0, qty=427), OrderBookLevel(price=63.9, qty=273), OrderBookLevel(price=63.8, qty=524), OrderBookLevel(price=63.7, qty=507), OrderBookLevel(price=63.6, qty=880)], ask=[OrderBookLevel(price=64.1, qty=90), OrderBookLevel(price=64.2, qty=170), OrderBookLevel(price=64.3, qty=444), OrderBookLevel(price=64.4, qty=266), OrderBookLevel(price=64.5, qty=694)], simulate=0)
[ALL] Best5:: 64.0, 427, Best5(code='2609', bid=[OrderBookLevel(price=64.0, qty=427), OrderBookLevel(price=63.9, qty=289), OrderBookLevel(price=63.8, qty=524), OrderBookLevel(price=63.7, qty=507), OrderBookLevel(price=63.6, qty=880)], ask=[OrderBookLevel(price=64.1, qty=90), OrderBookLevel(price=64.2, qty=170), OrderBookLevel(price=64.3, qty=444), OrderBookLevel(price=64.4, qty=266), OrderBookLevel(price=64.5, qty=694)], simulate=0)
[ALL] Best5:: 64.0, 422, Best5(code='2609', bid

### 4.1.4. 成交明細 (Tick)

訂閱 K 線也與行情相同，訂閱事件回傳即為 K 線物件 (Kline)

> <br/>
> 建置中<br/>
> <br/>

------

## 4.2.歷史行情

能索取的歷史行情包含 **成交明細 (Tick)** 與 **K 線 (KLine)**

### 4.2.1. 歷史成交明細

成交明細僅提供當日交易日的歷史數據，取得方式可透過呼叫函數和訂閱事件來取得

第一個方式：透過呼叫函數來取得歷史成交明細：

In [1]:
user_id = "A123456789"
password = "*************"

In [2]:
from eskmo import api
from eskmo import Stock

api.logger.show = False

@api.event.tick.history_notify
def onTickHistoryNotify(tick):
    print(f"[ALL] Tick history:: {tick}")

api.init()
api.login(userId=user_id, password=password)

stock: Stock = api.stocks["2330"]
stock.tick_history(isAsync=True) 

api.keepalive()

%BP INFO  2024-08-16 11:58:02.263591 | Main | 40784 |[ eskmo version: 0.0.88 ]


: 